In [ ]:
# 在 Google Colab 里运行时，先把本书的代码拉下来（本地 Jupyter 会跳过这一格）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/pytorch-inside-out"):
        !git clone -q --depth 1 https://github.com/wikieden/pytorch-inside-out /content/pytorch-inside-out
    %cd /content/pytorch-inside-out/code/notebooks


> 在线讲义：[第 22 章 · 分布式存档与排错：每卡存一片，卡死先查顺序](https://wikieden.github.io/pytorch-inside-out/lessons/22-dcp-debug.html) · 本书《PyTorch 从里到外》

# 第 22 章 · 分布式存档与排错：每卡存一片，卡死先查顺序

本章回答：模型切在几十张卡上，怎么存档、怎么在不同卡数下读回来？分布式训练最常见的"卡住不动"是怎么回事，怎么查？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/22-dcp-debug.md`（讲解、动画、练习都在讲义里）。对应原文：官方文档 Distributed Checkpoint（DCP）、`torch.distributed.checkpoint.state_dict`（〔在线〕）；多机报告 §7.2 可靠性、§7.3 NCCL 与启动、§9「常见故障速查」（`distributed-pretraining/notes/03-engineering-report.md`）；Ultra-Scale Playbook §A1 Distributed Training Profiling（`ultrascale-playbook.md` 第 3624 行起）；本章 DCP 输出在本机 Gloo 2 进程上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import sys
sys.path.insert(0, "..")   # 子进程要能按名字导入函数，所以用本地模块

import os
import tempfile

from torch.distributed.checkpoint.state_dict import get_state_dict, set_state_dict
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard
import torch
import torch.distributed as dist
import torch.distributed.checkpoint as dcp
import torch.nn as nn

import ch22_dcp_debug as M          # 传给 run(...) 的函数从这里取
from dist_util import run
torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 DCP 存与读 | 每 rank 一个文件 | `dcp_roundtrip` | restored True；.metadata + 2 个 .distcp |
| 第三步 条件分支 | 调用顺序不一致 | `call_sequence(r, over, False)` | rank 0 多一个 stats2 |
| 第三步 先统一条件 | 一致 | `call_sequence(r, over, True)` | 两个 rank 相同 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `call_sequence`**（来自 `ch22_dcp_debug.py`）：每步调用了哪些集合通信（不真的通信，只列出顺序，用来比对两个 rank 是否一致）。

In [ ]:
def call_sequence(rank: int, over_threshold: dict, fixed: bool) -> list:
    """每步调用了哪些集合通信（不真的通信，只列出顺序，用来比对两个 rank 是否一致）。"""
    seq = []
    for step in (1, 2, 3):
        if fixed:
            seq.append(f"flag{step}")
        seq.append(f"grad{step}")
        hit = any(over_threshold[r][step] for r in over_threshold) if fixed else over_threshold[rank][step]
        if hit:
            seq.append(f"stats{step}")
    return seq

**常量 `CKPT`**（来自 `ch22_dcp_debug.py`）

In [ ]:
CKPT = os.path.join(tempfile.gettempdir(), "pytorch-deep-dive-ch22-ckpt")

**函数 `dcp_roundtrip`**（来自 `ch22_dcp_debug.py`）：下面要用到的小工具。

In [ ]:
def dcp_roundtrip(rank, world):
    mesh = init_device_mesh("cpu", (world,))
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(4, 6), nn.Linear(6, 2))
    for layer in m:
        fully_shard(layer, mesh=mesh)
    fully_shard(m, mesh=mesh)
    opt = torch.optim.AdamW(m.parameters(), lr=0.1)
    m(torch.ones(1, 4)).sum().backward()
    opt.step()
    msd, osd = get_state_dict(m, opt)
    dcp.save({"model": msd, "optim": osd}, checkpoint_id=CKPT)       # 所有 rank 都要调用
    w0 = m[0].weight.full_tensor().clone()                           # full_tensor 也是集合通信
    with torch.no_grad():
        m[0].weight.zero_()
    msd, osd = get_state_dict(m, opt)
    state = {"model": msd, "optim": osd}
    dcp.load(state, checkpoint_id=CKPT)
    set_state_dict(m, opt, model_state_dict=state["model"], optim_state_dict=state["optim"])
    ok = torch.equal(m[0].weight.full_tensor(), w0)
    dist.barrier()
    return dict(restored=ok, files=sorted(os.listdir(CKPT)) if rank == 0 else None)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 DCP 存一遍、读回来：")
for r, res in enumerate(run(M.dcp_roundtrip)):
    print(f"  rank {r}: {res}")
over = {0: {1: False, 2: True, 3: False}, 1: {1: False, 2: False, 3: False}}

In [ ]:
print("\n第三步 条件依赖本 rank：", call_sequence(0, over, False), "vs", call_sequence(1, over, False))
print("先统一条件：", call_sequence(0, over, True), "vs", call_sequence(1, over, True))

## 3. 代码里才看得到的

1. **`call_sequence` 不真的通信**：真去跑错的版本会 hang 住测试，所以只列出调用顺序来比对。
2. **存档写到临时目录**：路径在 `CKPT`，重复运行会覆盖。

## 4. 自己改一改

- 把 `over` 里 rank 1 的第 3 步也改成 True，看错开的位置。
- `run(dcp_roundtrip, world=4)` 看文件数。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 分布式存档每卡存一片，元数据记位置，换卡数照样读；save、load 人人都要调；
> 卡死多半是通信次数顺序不一致：条件分支、提前退出、只让 rank 0 调；
> 先设超时，再开 DETAIL，py-spy 看停在哪，NCCL_DEBUG 看网络。